### Setup

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, ExperimentDataPreprocessor, set_seed
from common.eval import eval_user_diversity_preference_scale

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (0/0):
Data count before: 480608
Data count after: 480608
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480608
Num of distinct users: 2103
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480608
interaction data count after merging: 478564
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[bhiravi_vaidhy, dilip_satgare, haresh_mehta, ...",USA,siddharth_randeria,Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,mel_gibson,Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[dylan_walsh, laura_linney, ernie_hudson_jr, t...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358129 (74.83%
valid: 46916 (9.8%)
test: 73519 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555886
1    0.444114
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585155
1    0.414845
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358129
  Num of positive interactions: 159050 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 159050], edge_label=[159050])
Edge Index: tensor([[   0,    0,    0,  ..., 2093, 2093, 2093],
        [1102, 1186,  670,  ..., 2835,  742, 2969]])


#### Evaluate User Diversity Preference Scale

In [8]:
user_dps_df = eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True)
user_dps_df.head()

Calculating user diversity preference scale:   0%|          | 0/2094 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2094/2094 [00:32<00:00, 65.05it/s]


,userID,actorID_dist,actorID_dps,country_dist,country_dps,directorID_dist,directorID_dps,genre_dist,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.511464,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.159621,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.424931,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545
1,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.475977,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.300297,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.383176,"[15.0, 27.0, 12.0, 3.5, 30.0, 7.0, 0.0, 54.5, ...",0.764622
2,2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 0.0, 0.0, ...",0.485033,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.098869,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.384691,"[26.0, 11.0, 0.0, 3.0, 34.5, 6.5, 1.0, 22.5, 1...",0.793320
3,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.562356,"[0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, ...",0.111710,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.470833,"[55.5, 42.5, 9.0, 9.5, 91.5, 39.5, 0.0, 64.5, ...",0.814879
4,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.518630,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.297905,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.429368,"[44.5, 46.0, 0.0, 0.0, 35.0, 24.0, 0.0, 110.0,...",0.756129


In [9]:
encoded_train_df_with_dps = encoded_train_df.merge(user_dps_df, on="userID", how="left")

#### Prepare prediction pool for inference/testing

In [10]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2095
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2095(users) * 500(items) = 1047500


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1047495,2094,7957,0,"[5403, 102, 5844, 14185, 8714]",63,955,"[8, 15, 0, 0, 0, 0, 0, 0]"
1047496,2094,1611,0,"[13198, 4752, 15371, 5573, 8531]",63,2910,"[4, 5, 0, 0, 0, 0, 0, 0]"
1047497,2094,4715,0,"[2653, 2800, 5578, 9181, 11244]",63,1393,"[7, 18, 0, 0, 0, 0, 0, 0]"
1047498,2094,1924,0,"[3069, 7855, 11716, 8412, 5873]",62,2293,"[5, 0, 0, 0, 0, 0, 0, 0]"
1047499,2094,1590,0,"[6359, 5740, 13573, 3459, 14058]",63,2954,"[1, 2, 8, 0, 0, 0, 0, 0]"


### Prepare DataLoader

In [11]:
# TODO: determine which Dataset to use
from common.datasets import UserItemPairDataset
from torch.utils.data import DataLoader

BATCH_SIZE = 1024

train_dataset = UserItemPairDataset(encoded_train_df_with_dps)
valid_dataset = UserItemPairDataset(encoded_valid_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 358129
valid data count: 46916
test data count: 1047500


### Configure Model (LightningModule)

In [ ]:
from DPRecSys.models.gcn_cf_rec import GCNRecCF

EMB_DIM = 64
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 2

model = GCNRecCF(
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,  # shape [2, num_edges]
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
)


### Configure Trainer and Experiment

In [11]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "gcn-bce-exp"
RUN_NAME = "gcn-baseline-test4"
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME)
trainer_callbacks = get_callbacks(EXPERIMENT_NAME, RUN_NAME, patience=3)

In [12]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=1,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [13]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


You are using a CUDA device ('NVIDIA GeForce RTX 4070 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/emma/10TB/home/bilab_archive/Bai/DPRecSys/experiments/test_checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


🏃 View run gcn-baseline-test4 at: http://140.112.106.216:3683/#/experiments/3/runs/84113a46dc43461aabc82e217757e76d
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/3


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 11.72 GiB of which 1.75 MiB is free. Process 281845 has 3.55 GiB memory in use. Process 286051 has 7.40 GiB memory in use. Process 422585 has 182.00 MiB memory in use. Process 451268 has 274.00 MiB memory in use. Including non-PyTorch memory, this process has 182.00 MiB memory in use. Of the allocated memory 850.00 KiB is allocated by PyTorch, and 1.17 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Inference

In [15]:
# NOTE: the inference model MUST be the same as the training model
best_model_path = "test_checkpoints/gcn-bce-exp-gcn-baseline-test3-best-checkpoint-epoch=00-val_f1=0.57.ckpt"

model = GCNRecCF.load_from_checkpoint(
    checkpoint_path=best_model_path,
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,
    dim_id=EMB_DIM,
    num_layers=3,
    concat=True,
    lr=LR,
)


In [16]:
# start inference
trainer.test(model=model, dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.5445107221603394     │
│          test_f1          │    0.09266123175621033    │
│         test_loss         │     12384.9208984375      │
│         test_prec         │   0.049183208495378494    │
│         test_rec          │    0.7988131046295166     │
└───────────────────────────┴───────────────────────────┘

🏃 View run gcn-baseline-test3 at: http://140.112.106.216:3683/#/experiments/3/runs/8e967c6480084a9fa4f127c509d32687
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/3


[{'test_loss': 12384.9208984375,
  'test_acc': 0.5445107221603394,
  'test_prec': 0.049183208495378494,
  'test_rec': 0.7988131046295166,
  'test_f1': 0.09266123175621033}]

- gcn-bce-exp-gcn-baseline-test2-best-checkpoint-epoch=04-val_f1=0.45.ckpt
<details>
K=500

- [{'test_loss': 364.225341796875,
- 'test_acc': 0.7884315252304077,
- 'test_prec': 0.05633168667554855,
- 'test_rec': 0.39781633019447327,
- 'test_f1': 0.09868881106376648}]

K=250

- [{'test_loss': 372.6000061035156,
- 'test_acc': 0.7848190665245056,
- 'test_prec': 0.07433757930994034,
- 'test_rec': 0.3977948725223541,
- 'test_f1': 0.1252661496400833}]
</details>

- gcn-bce-exp-gcn-baseline-test3-best-checkpoint-epoch=00-val_f1=0.57.ckpt
<details>
K=500

- [{'test_loss': 12384.9208984375,
- 'test_acc': 0.5445107221603394,
- 'test_prec': 0.049183208495378494,
- 'test_rec': 0.7988131046295166,
- 'test_f1': 0.09266123175621033}]
</details>

In [17]:
model.test_results

{'user': tensor([   0,    0,    0,  ..., 2094, 2094, 2094]),
 'item': tensor([7977, 4839,  979,  ..., 4715, 1924, 1590]),
 'score': tensor([7.2073e+03, 3.9785e+02, 2.5718e+04,  ..., 1.2598e+00, 5.9260e+00,
         1.2725e+01]),
 'label': tensor([1., 0., 1.,  ..., 0., 0., 0.]),
 'metric': {'test_loss': 12384.9208984375,
  'test_acc': 0.5445107398568019,
  'test_prec': 0.04918320709313781,
  'test_rec': 0.7988130758385521,
  'test_f1': 0.09266122913144598},
 'user_emb': tensor([[-8.5692e+01,  3.2615e+03, -2.0484e+01,  ..., -1.9593e+01,
          -4.3632e+01, -3.4823e+00],
         [-4.0915e+01,  1.5595e+03, -9.7745e+00,  ..., -9.4607e+00,
          -2.0874e+01, -1.6690e+00],
         [-7.3165e+00,  2.7686e+02, -1.7549e+00,  ..., -1.6860e+00,
          -3.7386e+00, -3.0014e-01],
         ...,
         [-2.3794e+02,  9.0519e+03, -5.6874e+01,  ..., -5.4493e+01,
          -1.2116e+02, -9.6405e+00],
         [-4.3312e+02,  1.6519e+04, -1.0338e+02,  ..., -9.9769e+01,
          -2.2064e+02, -1